# 2. Which process effects can the ensemble measure?

Four tests, from cheapest to most direct. The first two run here in seconds; the last two read results
cached by `scripts/precompute.py`.

1. **Budget**: how much of the spread between models do the flags explain?
2. **Power**: how many models would be needed to detect an effect of a given size?
3. **Synthetic recovery**: plant known effects in pseudo-data on the real flags; are they found?
4. **Leave one out**: let each model play the real world; is it predicted better than by the raw ensemble?

In [ ]:
import pandas as pd
from tcre_evidence import process_plots as pp, process_tests as pt
from tcre_evidence.paths import DATA

pd.set_option("display.precision", 2)
gamma, flags, family = pt.load_ensemble(DATA / "examples" / "cmip6_land_feedbacks.csv",
                                        DATA / "examples" / "cmip6_process_flags.csv", "gamma_land_coubgc")
res = pp.load("cmip6", "gamma_land_coubgc")

## 1. Budget

A straight-line regression of γ_L on the flags. `r2` is the share of the spread the flags explain;
`loo_q2` is the same share when each model is predicted from the others. A negative `loo_q2` means the
flags predict a new model worse than simply using the ensemble mean.

In [ ]:
b = pt.flag_budget(gamma, flags)
print(f"in sample: {b['r2']:.0%} of the spread explained (adjusted for the number of flags: {b['adj_r2']:.2f})")
print(f"each model predicted from the others: {b['loo_q2']:.2f}")
print(f"spread left unexplained: {b['resid_sd']:.1f} PgC/K, against a total spread of {b['sd']:.1f} PgC/K")

## 2. Power

Given that unexplained spread, how many independent models would be needed to detect each effect at
two standard errors? The effect sizes here are those used in the synthetic test below (half the spread
between models).

In [ ]:
pp.power(res["power"]);
res["power"]

## 3. Synthetic recovery

Pseudo-ensembles are made on the real flag table, with the same scatter as the real data, in two
scenarios: every process has an effect, or none does. The process model is then refitted to each.

If an effect is identified, the estimate follows the truth: the points lie on the dashed 1:1 line. If it is
not, the estimate stays near its prior median (dotted line) whichever scenario is true.

In [ ]:
pp.recovery(res);

The **evidence table** sums up the three tests for each process. `tracking` is how far the estimate
moves when the truth moves (1 = follows the truth, 0 = ignores it).

In [ ]:
ev = pt.evidence_table(res["prior_vs_posterior"], res["synthetic_effects_summary"],
                       res["synthetic_no_effects_summary"], res["power"])
ev

## 4. Leave one model out

Each model in turn plays the real world: it (or its whole family) is removed, the model is refitted, and
three predictions of its γ_L are compared with its actual value:

* **raw ensemble**: the training models' mean and spread;
* **all processes on**: the real-world estimate, which is what the configuration reports for the real world;
* **own flags**: the prediction for a model with the held-out model's flags.

`crps_skill` below 0 means worse than the raw ensemble. The all-processes-on estimate is not expected to
match a model that lacks some processes; the point is that nothing in the ensemble can check the
adjustment it makes.

In [ ]:
pp.loo(res, "PgC/K");
res["loo_summary"]

## β_L

The same evidence table and leave-one-out summary for the land carbon-concentration feedback, where the
nitrogen effect is larger relative to the spread.

In [ ]:
res_b = pp.load("cmip6", "beta_land_coubgc")
pp.recovery(res_b);
pt.evidence_table(res_b["prior_vs_posterior"], res_b["synthetic_effects_summary"],
                  res_b["synthetic_no_effects_summary"], res_b["power"])

In [ ]:
res_b["loo_summary"]

## Reading the results together

A process effect is supported by the ensemble when the power test says the ensemble is large enough, the
synthetic test tracks the truth, the posterior differs from the prior, and the flagged models are not all
from one family. Where these fail, the effect in the result comes from the prior and is better stated as a
judgement. A failed test does not show that a process is unimportant, only that this ensemble cannot
measure it (calibration can also hide a real effect; see `scripts/m9_tuning_selection.py`).